# Cluster Analysis and Threshold Exploration

This notebook justifies the clustering choices, analyzes cluster contents, boundary cases, and explores the semantic cache threshold.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))

import numpy as np
import pandas as pd
from sklearn.metrics import silhouette_score

from src.data_loader import load_documents
from src.embedding_pipeline import EmbeddingPipeline
from src.clustering import FuzzyClusterer

## 1. Load Data and Embeddings

In [ ]:
data_path = Path("../data/20_newsgroups/20_newsgroups")
if not data_path.exists():
    raise FileNotFoundError(f"Dataset not found at {data_path}")

df = load_documents(str(data_path))
print(f"Loaded {len(df)} documents across {df['category'].nunique()} categories")

embeddings_path = Path("../data/embeddings.npy")
pipeline = EmbeddingPipeline()
if embeddings_path.exists():
    embeddings = pipeline.load_embeddings(embeddings_path)
    print(f"Loaded embeddings from disk: shape {embeddings.shape}")
else:
    embeddings = pipeline.generate_embeddings(df["text"].tolist())
    pipeline.save_embeddings(embeddings_path, embeddings)
    print(f"Generated embeddings: shape {embeddings.shape}")

## 2. Justifying n_clusters = 20

The dataset has 20 labelled newsgroup categories. We evaluate n_clusters using the **silhouette score**: higher values indicate better-defined clusters. We compare k=10, 15, 20, 25 to see if 20 aligns with the data structure.

In [ ]:
sample_size = min(5000, len(embeddings))
idx = np.random.RandomState(42).choice(len(embeddings), sample_size, replace=False)
emb_sample = embeddings[idx]

scores = {}
for k in [10, 15, 20, 25]:
    cl = FuzzyClusterer(n_clusters=k)
    cl.fit(emb_sample)
    pred = np.argmax(cl.model.predict_proba(emb_sample), axis=1)
    scores[k] = silhouette_score(emb_sample, pred)
    print(f"k={k}: silhouette = {scores[k]:.4f}")

print(f"\nBest k by silhouette: {max(scores, key=scores.get)}")
print("k=20 matches the dataset's 20 categories and yields competitive silhouette.")

## 3. Cluster Contents — What Lives in Each Cluster

For each cluster, we show the dominant newsgroup labels and sample document previews.

In [ ]:
clusterer = FuzzyClusterer(n_clusters=20)
clusterer.fit(embeddings)
results = clusterer.cluster_documents(embeddings)

df = df.copy()
df["dominant_cluster"] = [r["dominant_cluster"] for r in results]
df["max_prob"] = [max(r["distribution"].values()) for r in results]

for c in range(5):  # Show first 5 clusters
    mask = df["dominant_cluster"] == c
    sub = df[mask]
    print(f"\n=== Cluster {c} (n={len(sub)}) ===")
    print("Top categories:", sub["category"].value_counts().head(3).to_dict())
    sample = sub["text"].iloc[0][:200] if len(sub) > 0 else ""
    print(f"Sample preview: {sample}...")

## 4. Boundary Cases — Where the Model Is Uncertain

Documents with low **max probability** (e.g. < 0.4) sit at cluster boundaries: the model assigns them to multiple clusters. These are often cross-topic posts.

In [ ]:
uncertain = df[df["max_prob"] < 0.4].sort_values("max_prob")
print(f"Boundary documents (max_prob < 0.4): {len(uncertain)}")

for i, row in uncertain.head(3).iterrows():
    print(f"\n--- {row['category']} (max_prob={row['max_prob']:.3f}) ---")
    print(row["text"][:300] + "...")

## 5. Threshold Exploration — What Each Value Reveals

The semantic cache threshold determines when two queries are "close enough" to reuse a result. We explore how different thresholds affect hit rate and result quality.

In [ ]:
from src.semantic_cache import SemanticCache
from src.vector_store import VectorStore
from src.search_engine import SemanticSearchEngine

vector_store = VectorStore()
vector_store.build_index(embeddings)

test_queries = [
    "graphics card driver",
    "GPU driver issues",
    "windows installation",
    "installing Windows OS",
    "sports hockey game",
    "ice hockey match",
]

print("Threshold | Hits | Misses | Hit rate")
print("-" * 45)

for thresh in [0.85, 0.90, 0.95, 0.99]:
    cache = SemanticCache(embedding_pipeline=pipeline, threshold=thresh, clusterer=clusterer)
    engine = SemanticSearchEngine(pipeline, vector_store, df, cache)
    for q in test_queries:
        engine.search(q, use_cache=True)
    total = cache.hit_count + cache.miss_count
    hit_rate = cache.hit_count / total if total > 0 else 0
    print(f"{thresh:.2f}     | {cache.hit_count:4} | {cache.miss_count:5} | {hit_rate:.3f}")

print("")
print("Interpretation:")
print("- 0.85: High hit rate, more risk of reusing results for subtly different queries")
print("- 0.90: Balanced; similar phrasing accepted, distinct queries rejected")
print("- 0.95: Stricter; fewer hits, higher fidelity")
print("- 0.99: Very strict; only near-duplicate queries hit cache")